# NFL Draft Round 1 Skill Positions — 1995–2025
Database of every QB, RB, WR, TE taken in Round 1 from 1995 through 2025 (318 players).

## Setup

In [ ]:
import pandas as pd
from pathlib import Path

CSV = Path("nfl_draft_r1_skill_1995_2025.csv")
df = pd.read_csv(CSV)
print(f"{len(df)} players loaded")
df.head()

## Overview — picks per year by position

In [ ]:
pivot = (
    df.groupby(["year", "position"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["QB", "RB", "WR", "TE"])
)
pivot["Total"] = pivot.sum(axis=1)
pivot

## Visualization — R1 skill picks per year (stacked bar)

In [ ]:
import matplotlib.pyplot as plt

colors = {"QB": "#e74c3c", "RB": "#3498db", "WR": "#2ecc71", "TE": "#f39c12"}
ax = pivot[["QB","RB","WR","TE"]].plot(
    kind="bar", stacked=True, figsize=(16, 5),
    color=[colors[c] for c in ["QB","RB","WR","TE"]],
    edgecolor="none"
)
ax.set_title("NFL Round 1 Skill Position Picks Per Year (1995–2025)", fontsize=14)
ax.set_xlabel("Draft Year")
ax.set_ylabel("# Picks")
ax.set_xticklabels(pivot.index, rotation=45, ha="right")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## Query helpers

In [ ]:
def picks_by_year(year):
    return df[df.year == year].sort_values("pick").reset_index(drop=True)

def picks_by_pos(pos):
    return df[df.position == pos.upper()].sort_values(["year","pick"]).reset_index(drop=True)

def find_player(name):
    return df[df.name.str.contains(name, case=False)].reset_index(drop=True)

def picks_by_college(college):
    return df[df.college.str.contains(college, case=False)].sort_values(["year","pick"]).reset_index(drop=True)

def picks_by_team(team):
    return df[df.team.str.contains(team, case=False)].sort_values(["year","pick"]).reset_index(drop=True)

# Example usage — uncomment to run:
# picks_by_year(2011)
# picks_by_pos("WR")
# find_player("Peterson")
# picks_by_college("Alabama")
# picks_by_team("Cowboys")

## Browse by year — change the number to explore

In [ ]:
picks_by_year(1995)

## Top colleges by R1 skill position picks

In [ ]:
df.groupby("college").size().sort_values(ascending=False).head(20).to_frame("R1_picks")

## Injury analysis — cross-reference with data.js verdicts

Players that appear in both this database and the injury analysis dataset (`data.js`) are annotated below with their verdict (CONFIRMED / REFUTED / MIXED / NOFLAG).

In [ ]:
# Players currently in the injury analysis (from data.js v2.9)
# Format: (name_fragment, year, verdict)
ANALYZED = [
    ("Ki-Jana Carter",       1995, "CONFIRMED"),
    ("Jamal Lewis",          2000, "MIXED"),
    ("Willis McGahee",       2003, "REFUTED"),
    ("Charles Rogers",       2003, "NOFLAG"),
    ("Cadillac Williams",    2005, "CONFIRMED"),
    ("Greg Olsen",           2007, "CONFIRMED"),   # placeholder — verify
    ("Matt Forte",           2008, "REFUTED"),
    ("Michael Crabtree",     2009, "CONFIRMED"),
    ("Demaryius Thomas",     2010, "CONFIRMED"),
    ("Sam Bradford",         2010, "CONFIRMED"),
    ("Julio Jones",          2011, "REFUTED"),
    ("Mark Ingram",          2011, "REFUTED"),
    ("Justin Hunter",        2013, "CONFIRMED"),
    ("Saquon Barkley",       2018, "NOFLAG"),
    ("Drake London",         2022, "REFUTED"),
    ("Jameson Williams",     2022, "MIXED"),
]

verdict_map = {name: v for name, _, v in ANALYZED}

def get_verdict(name):
    for key, v in verdict_map.items():
        if key.lower() in name.lower() or name.lower() in key.lower():
            return v
    return "—"

df["verdict"] = df["name"].apply(get_verdict)
analyzed = df[df.verdict != "—"].sort_values(["year","pick"])
print(f"{len(analyzed)} players analyzed out of {len(df)} total ({len(analyzed)/len(df)*100:.1f}%)")
analyzed[["year","pick","position","name","team","verdict"]]

## Years not yet researched

Shows which draft years have zero players in the injury analysis — the next targets for research.

In [ ]:
covered_years = set(year for _, year, _ in ANALYZED)
all_years = sorted(df.year.unique())
gaps = [y for y in all_years if y not in covered_years]
print("Years with NO players researched yet:")
print(gaps)